# PolyVoice Engine — Chatterbox TTS on Colab GPU

Runs the **multilingual voice-cloning engine on Google's GPU** and exposes it on a public URL.
Nothing runs on your laptop — your RAM is not touched.

## Before you start
1. Menu: **Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save**.
2. Run the cells **top to bottom** (Shift+Enter on each).
3. The last cell prints a `https://....trycloudflare.com` URL.
   - Open it in your browser → full Chatterbox UI, running on GPU. Clone voices + 23 languages.
   - OR paste it into `polyvoice/.env.local` as `CHATTERBOX_URL=...` and restart `npm run dev`.

Keep this Colab tab open — closing it stops the engine and the URL dies.

In [ ]:
# @title 1. Verify GPU is on (stops early with a clear message if not)
import torch
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise SystemExit("NO GPU. Fix: Runtime > Change runtime type > T4 GPU > Save, then re-run this cell.")

In [ ]:
# @title 2. Clone the engine + install dependencies (~3-5 min)
import os, subprocess, sys

if not os.path.isdir('engine'):
    !git clone -q https://github.com/mirbehnam/Chatterbox-TTS-Server-windows-easyInstallation.git engine
%cd /content/engine

# Multilingual Chatterbox (provides chatterbox.mtl_tts.ChatterboxMultilingualTTS)
!pip install -q chatterbox-tts
# Server deps (the repo's requirements are Windows-pinned; install the essentials directly)
!pip install -q fastapi "uvicorn[standard]" pyyaml soundfile librosa safetensors python-multipart \
    requests jinja2 aiofiles unidecode inflect tqdm pydub audiotsm watchdog

# cloudflared (public tunnel, no signup)
if not os.path.exists('cloudflared'):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

# IMPORTANT: confirm chatterbox install did not swap in a CPU-only torch
import torch
print("\nAfter install -> torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("CUDA got disabled by a dependency. Reinstalling a CUDA build of torch...")
    !pip install -q --force-reinstall torch torchaudio --index-url https://download.pytorch.org/whl/cu121
    print("Re-run THIS cell once more, then continue.")

In [ ]:
# @title 3. Point the engine at the GPU + fix Linux paths
import yaml
with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['tts_engine']['device'] = 'cuda'          # was 'cpu'
cfg['server']['host'] = '0.0.0.0'
cfg['server']['port'] = 8004
cfg['server']['use_ngrok'] = False
cfg['server']['log_file_path'] = 'logs/tts_server.log'   # forward slashes for Linux

with open('config.yaml', 'w') as f:
    yaml.safe_dump(cfg, f, sort_keys=False)
print("config.yaml patched -> device = cuda")

In [ ]:
# @title 4. Launch the engine + open the public tunnel
# Loads the base model (~30-60s on GPU), then prints a public URL. Keep this cell running.
import threading, time, subprocess, re, requests, uvicorn

def run_server():
    from server import app
    uvicorn.run(app, host='0.0.0.0', port=8004, log_level='warning', access_log=False)

threading.Thread(target=run_server, daemon=True).start()

print("Waiting for the engine to load the model...")
ready = False
for _ in range(180):  # up to 6 minutes
    try:
        if requests.get('http://localhost:8004/docs', timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
    time.sleep(2)

if not ready:
    raise SystemExit("Engine did not come up. Scroll up for the traceback, fix, and re-run.")
print("Engine is up. Opening public tunnel...\n")

proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:8004', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
public_url = None
for line in proc.stdout:
    print(line, end='')
    m = re.search(r'https://[-a-z0-9]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

print("\n\n================  ENGINE IS LIVE ON GPU  ================")
print("PUBLIC URL:", public_url)
print("")
print("Option A (zero local): open that URL in your browser -> Chatterbox UI.")
print("Option B (PolyVoice):  put this line in polyvoice/.env.local then run `npm run dev`:")
print(f"   CHATTERBOX_URL={public_url}")
print("========================================================")

# Keep this cell alive so the tunnel + engine stay up.
proc.wait()